## Aufgabe 1 · Dummy-Daten einfügen

Ein Projekt wird angelegt und mit verschiedenen Beispiel-Embeddings befüllt
(Fragen, Notizen, Code-Snippets). Die Ausgabe wird in einem scrollbaren
HTML-Bereich dargestellt, damit der Notebook-Output übersichtlich bleibt.

In [ ]:
dummy_data = [
    {"content": "Wie funktioniert Retrieval-Augmented Generation (RAG)?",         "metdata": {"type": "question", "topic": "RAG"}},
    {"content": "Embeddings sind Vektordarstellungen von Texten.",                 "metdata": {"type": "note",     "topic": "Embeddings"}},
    {"content": "def cosine_similarity(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))",
                                                                                    "metdata": {"type": "code",     "topic": "Python"}},
    {"content": "Was ist der Unterschied zwischen RAG und Fine-Tuning?",           "metdata": {"type": "question", "topic": "RAG"}},
    {"content": "ChromaDB ist eine Vektordatenbank für Embeddings.",               "metdata": {"type": "note",     "topic": "ChromaDB"}},
    {"content": "Wie speichere ich Embeddings in Supabase?",                       "metdata": {"type": "question", "topic": "Supabase"}},
]

project_id = add_project("Dummy-Projekt für Tests")
for item in dummy_data:
    add_embedding(project_id, item["content"], item["metdata"])

# Ausgabe scrollbar anzeigen
buf = io.StringIO()
with redirect_stdout(buf):
    embeddings = log_embeddings(project_id)
display(HTML(
    "<div style='height:200px;overflow:auto;border:1px solid #ccc;"
    "padding:10px;background:#f8f8f8'><pre>" + buf.getvalue() + "</pre></div>"
))

✅ Projekt 'Dummy-Projekt für Tests' erstellt (ID: 0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8)
✅ Embedding hinzugefügt: 'Wie funktioniert Retrieval-Augmented Gen...' (ID: a82205d4...)
✅ Embedding hinzugefügt: 'Embeddings sind Vektordarstellungen von ...' (ID: fbc97e42...)
✅ Embedding hinzugefügt: 'def cosine_similarity(a, b): return np.d...' (ID: dc475326...)
✅ Embedding hinzugefügt: 'Was ist der Unterschied zwischen RAG und...' (ID: f93690fd...)
✅ Embedding hinzugefügt: 'ChromaDB ist eine Vektordatenbank für Em...' (ID: 73b711e6...)
✅ Embedding hinzugefügt: 'Wie speichere ich Embeddings in Supabase...' (ID: 745c263c...)


## Aufgabe 2 · Text chunken und speichern

Lange Texte müssen vor der Einbettung in kleinere, überlappende Abschnitte
(*Chunks*) aufgeteilt werden. Die Überlappung (`overlap`) stellt sicher, dass
kein Kontext an den Chunk-Grenzen verloren geht.

> Experimentiere mit den Parametern `chunk_size` und `overlap`.

In [ ]:
long_text = """
Retrieval-Augmented Generation (RAG) is a technique that combines the strengths of
large language models (LLMs) with external knowledge bases. The core idea is to
retrieve relevant information from a database or document collection and then use
this information as context for the LLM to generate more accurate and up-to-date responses.
This approach is particularly useful when the LLM's training data might be outdated or
when the task requires domain-specific knowledge not present in the model's weights.
"""

chunks = chunk_text(long_text, chunk_size=100, overlap=20)
print(f"📝 Text in {len(chunks)} Chunks zerlegt:")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}: {chunk[:100]}")

chunk_project_id = add_project("Chunked Text – RAG Erklärung")
for i, chunk in enumerate(chunks):
    add_embedding(chunk_project_id, chunk, {"type": "chunk", "original_text": "RAG Erklärung", "chunk_index": i})
print(f"\n✅ {len(chunks)} Chunks gespeichert.")

📝 Text in 7 Chunks zerlegt:
  Chunk 1: 
Retrieval-Augmented Generation (RAG) is a technique that combines the strengths of
large language m...
  Chunk 2:  of
large language models (LLMs) with external knowledge bases. The core idea is to
retrieve relevan...
  Chunk 3:  to
retrieve relevant information from a database or document collection and then use
this informati...
  Chunk 4: n use
this information as context for the LLM to generate more accurate and up-to-date responses.
Th...
  Chunk 5: o-date responses.
This approach is particularly useful when the LLM's training data might be outdate...
  Chunk 6: ata might be outdated or
when the task requires domain-specific knowledge not present in the model's...
  Chunk 7: esent in the model's weights.
...
✅ Projekt 'Chunked Text – RAG Erklärung' erstellt (ID: 570676b5-63d4-4529-b1ea-a312b18e91a4)
✅ Embedding hinzugefügt: '
Retrieval-Augmented Generation (RAG) is...' (ID: fc1bfaf6...)
✅ Embedding hinzugefügt: ' of
large language models (LL

## Aufgabe 3 · Similarity-Suche

Über den Supabase RPC-Endpunkt `match_embeddings` wird eine
Cosine-Similarity-Suche gegen alle gespeicherten Embeddings des Nutzers
durchgeführt. Die `top_k` ähnlichsten Ergebnisse werden zurückgegeben.

> Probiere verschiedene Anfragen aus – auch themenfremde, um zu sehen, was
> das Modell als 'ähnlich' einschätzt.

In [ ]:
search_similar("Was ist RAG?",                   project_id=project_id)
search_similar("Wie funktionieren Embeddings?",   project_id=project_id)
search_similar("cosine similarity Python",        project_id=project_id)


🔎 Suche nach: 'Was ist RAG?'
  1. Was ist der Unterschied zwischen RAG und Fine-Tuning?... (Similarity: 0.61)
  2. 
Retrieval-Augmented Generation (RAG) is a technique that combines the strengths... (Similarity: 0.33)
  3. 
Retrieval-Augmented Generation (RAG) is a technique that combines the strengths... (Similarity: 0.33)

🔎 Suche nach: 'Wie funktionieren Embeddings?'
  1. Embeddings sind Vektordarstellungen von Texten.... (Similarity: 0.69)
  2. Wie speichere ich Embeddings in Supabase?... (Similarity: 0.61)
  3. ChromaDB ist eine Vektordatenbank für Embeddings.... (Similarity: 0.59)

🔎 Suche nach: 'cosine similarity Python'
  1. def cosine_similarity(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linal... (Similarity: 0.81)
  2. esent in the model's weights.
... (Similarity: 0.13)
  3. esent in the model's weights.
... (Similarity: 0.13)


[{'id': 'dc475326-fada-4117-bbb4-e8ebfeb21f7c',
  'project_id': '0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8',
  'content': 'def cosine_similarity(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))',
  'metdata': {'type': 'code', 'topic': 'Python'},
  'similarity': 0.814061544649423},
 {'id': '01187d11-8ccc-44f2-92aa-16f2ec4952f5',
  'project_id': '00075001-ce7d-405a-91c1-65100d364470',
  'content': "esent in the model's weights.\n",
  'metdata': {'type': 'chunk',
   'chunk_index': 6,
   'original_text': 'RAG Erklärung'},
  'similarity': 0.125012829898259},
 {'id': 'f1e8fd5e-9962-4326-9a64-5923a3a35431',
  'project_id': '570676b5-63d4-4529-b1ea-a312b18e91a4',
  'content': "esent in the model's weights.\n",
  'metdata': {'type': 'chunk',
   'chunk_index': 6,
   'original_text': 'RAG Erklärung'},
  'similarity': 0.125012829898259}]

## Aufgabe 4 · Daten löschen

Einzelne Embeddings können per ID entfernt werden. `delete_project` löscht
zunächst alle zugehörigen Embeddings und anschließend das Projekt selbst
(Fremdschlüssel-Reihenfolge).

> Hier wird das erste Embedding aus dem Dummy-Projekt gelöscht und der
> Zustand danach geloggt.

In [ ]:
embeddings = log_embeddings(project_id)
if embeddings:
    delete_embedding(embeddings[0]["id"])
    print("\nZustand nach dem Löschen:")
    log_embeddings(project_id)

📄 Embeddings in Projekt 0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8 (6):
  - Wie funktioniert Retrieval-Augmented Generation (R... (Quelle: unknown, Embedding: [-0.1331, 0.0228, -0.0391..., 0.0270, -0.0354, 0.0245])
  - Embeddings sind Vektordarstellungen von Texten.... (Quelle: unknown, Embedding: [-0.0717, -0.0205, 0.0255..., 0.0501, 0.0478, -0.0251])
  - def cosine_similarity(a, b): return np.dot(a, b) /... (Quelle: unknown, Embedding: [-0.0164, -0.0336, -0.1228..., 0.0153, 0.0687, -0.0192])
  - Was ist der Unterschied zwischen RAG und Fine-Tuni... (Quelle: unknown, Embedding: [-0.0368, 0.0294, 0.0250..., -0.0278, 0.0263, -0.0211])
  - ChromaDB ist eine Vektordatenbank für Embeddings.... (Quelle: unknown, Embedding: [-0.0795, 0.0223, -0.0472..., 0.0372, 0.0689, -0.0162])
  - Wie speichere ich Embeddings in Supabase?... (Quelle: unknown, Embedding: [-0.0291, -0.0331, -0.0292..., -0.0054, 0.0679, -0.0003])
🗑️  Embedding a82205d4-1055-458d-83cc-a330e54e2462 gelöscht.

Zustand nach dem Löschen

## Aufgabe 5 · Metadaten aktualisieren

Embeddings können nachträglich mit neuen Metadaten angereichert werden,
ohne den Inhalt oder den Vektor neu zu berechnen – z. B. um Tags, Prioritäten
oder Kategorien hinzuzufügen.

In [ ]:
embeddings = log_embeddings(project_id)
if embeddings:
    first = embeddings[0]
    enriched_metdata = {**first["metdata"], "tags": ["important", "example"]}
    update_embedding(first["id"], new_metdata=enriched_metdata)
    print("✅ Metadaten aktualisiert.")

📄 Embeddings in Projekt 0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8 (5):
  - Embeddings sind Vektordarstellungen von Texten.... (Quelle: unknown, Embedding: [-0.0717, -0.0205, 0.0255..., 0.0501, 0.0478, -0.0251])
  - def cosine_similarity(a, b): return np.dot(a, b) /... (Quelle: unknown, Embedding: [-0.0164, -0.0336, -0.1228..., 0.0153, 0.0687, -0.0192])
  - Was ist der Unterschied zwischen RAG und Fine-Tuni... (Quelle: unknown, Embedding: [-0.0368, 0.0294, 0.0250..., -0.0278, 0.0263, -0.0211])
  - ChromaDB ist eine Vektordatenbank für Embeddings.... (Quelle: unknown, Embedding: [-0.0795, 0.0223, -0.0472..., 0.0372, 0.0689, -0.0162])
  - Wie speichere ich Embeddings in Supabase?... (Quelle: unknown, Embedding: [-0.0291, -0.0331, -0.0292..., -0.0054, 0.0679, -0.0003])
🔄 Embedding fbc97e42-2e9a-4634-899b-4e8f5f1c9b46 aktualisiert.
✅ Metadaten aktualisiert.


## Aufgabe 6 · Benchmarking

Die Funktion `benchmark_search` misst die Latenz der Similarity-Suche
über mehrere Iterationen und gibt Durchschnitt, Minimum und Maximum aus.

> Vergleiche zwei inhaltlich unterschiedliche Anfragen.

In [1]:
benchmark_search("Was ist ein Embedding?")
benchmark_search("RAG vs Fine-Tuning")

NameError: name 'benchmark_search' is not defined

## Aufgabe 7 · Daten exportieren

Das erste Projekt wird als pandas DataFrame exportiert. So lassen sich
die Daten lokal weiterverarbeiten, sichern oder visualisieren.

In [ ]:
projects = log_projects()
if projects:
    df = export_project_data(projects[0]["id"])
    print(df[["content", "metdata"]].head())

Deine bisherigen Projekte (3):
  - Dummy-Projekt für Tests (ID: 0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8)
  - Chunked Text – RAG Erklärung (ID: 00075001-ce7d-405a-91c1-65100d364470)
  - Chunked Text – RAG Erklärung (ID: 570676b5-63d4-4529-b1ea-a312b18e91a4)
📥 Exportiert: 5 Embeddings
                                             content  \
0  def cosine_similarity(a, b): return np.dot(a, ...   
1  Was ist der Unterschied zwischen RAG und Fine-...   
2  ChromaDB ist eine Vektordatenbank für Embeddings.   
3          Wie speichere ich Embeddings in Supabase?   
4    Embeddings sind Vektordarstellungen von Texten.   

                                             metdata  
0                {'type': 'code', 'topic': 'Python'}  
1               {'type': 'question', 'topic': 'RAG'}  
2              {'type': 'note', 'topic': 'ChromaDB'}  
3          {'type': 'question', 'topic': 'Supabase'}  
4  {'tags': ['important', 'example'], 'type': 'no...  


## Aufgabe 8 · Assoziationen herstellen

Zwei Embeddings können bidirektional verknüpft werden. Die Verknüpfung
wird als `links`-Liste im `metdata`-Feld beider Embeddings gespeichert.
Damit lassen sich einfache Knowledge-Graph-Strukturen abbilden.

In [ ]:
embeddings = log_embeddings(project_id)
if len(embeddings) >= 2:
    link_embeddings(embeddings[0]["id"], embeddings[1]["id"], "related")

    # Assoziationen des ersten Embeddings anzeigen
    e1 = (
        supabase.table("mempalace_embeddings")
        .select("metdata").eq("id", embeddings[0]["id"]).single().execute()
    )
    print("Gespeicherte Links:", e1.data["metdata"].get("links", []))

📄 Embeddings in Projekt 0169de38-a8a4-4ba6-b29d-ca6a5d1a0cf8 (5):
  - def cosine_similarity(a, b): return np.dot(a, b) /... (Quelle: unknown, Embedding: [-0.0164, -0.0336, -0.1228..., 0.0153, 0.0687, -0.0192])
  - Was ist der Unterschied zwischen RAG und Fine-Tuni... (Quelle: unknown, Embedding: [-0.0368, 0.0294, 0.0250..., -0.0278, 0.0263, -0.0211])
  - ChromaDB ist eine Vektordatenbank für Embeddings.... (Quelle: unknown, Embedding: [-0.0795, 0.0223, -0.0472..., 0.0372, 0.0689, -0.0162])
  - Wie speichere ich Embeddings in Supabase?... (Quelle: unknown, Embedding: [-0.0291, -0.0331, -0.0292..., -0.0054, 0.0679, -0.0003])
  - Embeddings sind Vektordarstellungen von Texten.... (Quelle: unknown, Embedding: [-0.0717, -0.0205, 0.0255..., 0.0501, 0.0478, -0.0251])
🔄 Embedding dc475326-fada-4117-bbb4-e8ebfeb21f7c aktualisiert.
🔄 Embedding f93690fd-9862-4035-870b-b64645f33ec0 aktualisiert.
🔗 Assoziation 'related' zwischen dc475326-fada-4117-bbb4-e8ebfeb21f7c und f93690fd-9862-4035-870b-b6464

## Aufgabe 9 · Daten aggregieren

Alle Embeddings des ersten Projekts werden nach dem `topic`-Feld
in den Metadaten gruppiert und übersichtlich ausgegeben.

In [ ]:
aggregate_by_topic(project_id)

📊 Embeddings nach Themen:
  - **RAG** (1 Einträge)
    • Was ist der Unterschied zwischen RAG und Fine-Tuni...
  - **ChromaDB** (1 Einträge)
    • ChromaDB ist eine Vektordatenbank für Embeddings....
  - **Supabase** (1 Einträge)
    • Wie speichere ich Embeddings in Supabase?...
  - **Embeddings** (1 Einträge)
    • Embeddings sind Vektordarstellungen von Texten....
  - **Python** (1 Einträge)
    • def cosine_similarity(a, b): return np.dot(a, b) /...


---
## 🎉 Zusammenfassung

| Aufgabe | Beschreibung | Funktion(en) |
|---------|--------------|---------------|
| 1 | Dummy-Daten einfügen | `add_project`, `add_embedding` |
| 2 | Text chunken & speichern | `chunk_text`, `add_embedding` |
| 3 | Similarity-Suche | `search_similar` |
| 4 | Daten löschen | `delete_embedding`, `delete_project` |
| 5 | Metadaten aktualisieren | `update_embedding` |
| 6 | Benchmarking | `benchmark_search` |
| 7 | Daten exportieren | `export_project_data` |
| 8 | Assoziationen | `link_embeddings` |
| 9 | Aggregieren | `aggregate_by_topic` |

### 💡 Hinweise
- **Jeder Nutzer arbeitet isoliert** (Row Level Security in Supabase).
- **Embeddings werden automatisch generiert** (`all-MiniLM-L6-v2`, 384 Dimensionen).
- **Metadaten sind flexibel** – beliebige JSON-Felder wie `topic`, `type`, `tags`.